In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
from torchvision import transforms
import os


class MRIDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        # Auto-detect classes (alphabetical)
        class_names = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])

        self.class_to_idx = {
            name: i for i, name in enumerate(class_names)
        }

        print("CLASS MAPPING:")
        for k, v in self.class_to_idx.items():
            print(f"  {k} → {v}")

        for class_name in class_names:
            class_path = os.path.join(root_dir, class_name)

            for img_name in os.listdir(class_path):
                if img_name.lower().endswith(
                    ('.png', '.jpg', '.jpeg')
                ):
                    self.images.append(
                        os.path.join(class_path, img_name)
                    )
                    self.labels.append(
                        self.class_to_idx[class_name]
                    )

        print(f"Loaded {len(self.images)} images from {root_dir}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )
        print("Done!!!")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

train_dataset = MRIDataset(
    '/content/drive/MyDrive/Colab Notebooks/train',
    transform=transform
)

test_dataset = MRIDataset(
    '/content/drive/MyDrive/Colab Notebooks/test',
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("\nDATA LOADERS READY!")

CLASS MAPPING:
  Mild Impairment → 0
  Moderate Impairment → 1
  No Impairment → 2
  Very Mild Impairment → 3
Loaded 10255 images from /content/drive/MyDrive/Colab Notebooks/train
CLASS MAPPING:
  Mild Impairment → 0
  Moderate Impairment → 1
  No Impairment → 2
  Very Mild Impairment → 3
Loaded 1279 images from /content/drive/MyDrive/Colab Notebooks/test

DATA LOADERS READY!


In [ ]:
!pip install timm -q

import timm
import torch
import torch.nn as nn
import torch.optim as optim

# Load EfficientNet-B0 pretrained on ImageNet
model = timm.create_model(
    'mobilenetv3_small_100',
    pretrained=True,
    num_classes=4
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Lower learning rate for transfer learning
optimizer = optim.Adam(
    model.parameters(),
    lr=0.0003
)

print("mobilenetv3_small_100 loaded! Ready for training.")
print(f"Using device: {device}")

model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

mobilenetv3_small_100 loaded! Ready for training.
Using device: cpu


In [ ]:
print("\nSTARTING TRAINING (mobilenetv3_small_100 - 2 EPOCHS)...")
for epoch in range(2):
    model.train()
    running_loss = 0.0
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if i % 100 == 0:
            print(f"  [Epoch {epoch+1}] Step {i}: Loss = {loss.item():.4f}")

    avg_loss = running_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} → Avg Loss: {avg_loss:.4f}")

print("\nTRAINING COMPLETE!")



STARTING TRAINING (EfficientNet-B0 - 2 EPOCHS)...
  [Epoch 1] Step 0: Loss = 2.9174
  [Epoch 1] Step 100: Loss = 0.9555
  [Epoch 1] Step 200: Loss = 0.5464
  [Epoch 1] Step 300: Loss = 0.5402

Epoch 1 → Avg Loss: 0.8263
  [Epoch 2] Step 0: Loss = 1.0835
  [Epoch 2] Step 100: Loss = 0.5499
  [Epoch 2] Step 200: Loss = 0.3840
  [Epoch 2] Step 300: Loss = 0.4585

Epoch 2 → Avg Loss: 0.4607

TRAINING COMPLETE!


In [ ]:
 model.eval()
 correct = total = 0
 with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        pred = out.argmax(1)
        correct += (pred == y).sum().item()
        total += y.size(0)
 accuracy = 100 * correct / total
 print(f"\n FINAL ACCURACY: {accuracy:.2f}%")
 print(f"Correct: {correct} / {total}")


 FINAL ACCURACY: 67.71%
Correct: 866 / 1279


In [ ]:
torch.save(
    model.state_dict(),
    '/content/mobilenetv3_small_100_final.pth'
)

!mkdir -p "/content/drive/MyDrive/Alzheimer_MVP"

!cp /content/mobilenetv3_small_100_final.pth "/content/drive/MyDrive/Alzheimer_MVP/"

print("✅ MODEL SAVED PERMANENTLY!")

✅ MODEL SAVED PERMANENTLY!


In [ ]:
# ============================
# INSTALL DEPENDENCIES
# ============================

!pip install -q streamlit pyngrok timm

# ============================
# COPY MODEL FROM DRIVE
# ============================

!cp "/content/drive/MyDrive/Alzheimer_MVP/mobilenetv3_small_100_final.pth" /content/

# ============================
# CREATE STREAMLIT APP
# ============================

app_code = '''
import streamlit as st
import torch
from PIL import Image
from torchvision import transforms
import timm

@st.cache_resource
def load_model():

    model = timm.create_model(
        "mobilenetv3_small_100",
        pretrained=False,
        num_classes=4
    )

    model.load_state_dict(
        torch.load(
            "/content/mobilenetv3_small_100_final.pth",
            map_location="cpu"
        )
    )

    model.eval()

    return model


model = load_model()

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

classes = [
    "Mild Impairment",
    "Moderate Impairment",
    "No Impairment",
    "Very Mild Impairment"
]

st.set_page_config(
    page_title="Alzheimer MRI AI",
    page_icon="🧠"
)

st.title("🧠 Alzheimer MRI Classification")

st.write("Upload an MRI image to predict Alzheimer's severity.")

uploaded = st.file_uploader(
    "Upload MRI Image",
    type=["jpg","jpeg","png"]
)

if uploaded:

    img = Image.open(uploaded).convert("RGB")

    st.image(
        img,
        caption="Uploaded MRI",
        use_container_width=True
    )

    x = transform(img).unsqueeze(0)

    with torch.no_grad():

        probs = torch.softmax(
            model(x),
            dim=1
        )[0]

    pred_idx = probs.argmax().item()

    st.success(
        f"Prediction: {classes[pred_idx]}"
    )

    st.subheader("Confidence Scores")

    st.bar_chart({
        classes[i]: float(probs[i])
        for i in range(4)
    })
'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("app.py created successfully")


# ============================
# STOP OLD PROCESSES
# ============================
!pkill -f streamlit || true
!pkill -f ngrok || true

# ============================
# START STREAMLIT (IMPORTANT FIX)
# ============================
!nohup streamlit run /content/app.py \
--server.port 8501 \
--server.address 0.0.0.0 \
--server.headless true \
> log.txt 2>&1 &

# ============================
# WAIT FOR STARTUP
# ============================
import time
time.sleep(20)

# ============================
# NGROK SETUP
# ============================
from pyngrok import ngrok

ngrok.set_auth_token("3ElzitRl1hHjarJtXdMnUN6NTc7_7XvKXKjuexyVBVz6yxoSe")

# kill old tunnels (important)
ngrok.kill()

tunnel = ngrok.connect(8501)

print("\n================================")
print("🔥 LIVE DEMO READY")
print("================================")
print(tunnel.public_url)

# ============================
# LOGS
# ============================
print("\nSTREAMLIT LOGS:\n")
!tail -30 log.txt

app.py created successfully
^C
^C

🔥 LIVE DEMO READY
https://sly-prance-headstand.ngrok-free.dev

STREAMLIT LOGS:



2026-06-06 18:09:02.473 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.151.234:8501

